# ตอนที่ 5: GRPO — ลบ value network ทิ้ง แล้วให้กลุ่มคำตอบเป็น baseline ของกันเอง

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/05_grpo.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 5/10] GRPO: ลบ value network ทิ้ง แล้วให้กลุ่มคำตอบเป็น baseline ของกันเอง**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-05-grpo)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation)
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline ก่อนแตะอะไรทั้งสิ้น
6. เตรียมข้อมูล (Data)
7. โค้ดหลัก (Main code) — **คิด group advantage เองด้วยมือ แล้ว assert ว่าตรงกับสูตร**
8. ผลลัพธ์ (Results)
9. เปรียบเทียบ (Comparison) — pass@1 vs pass@8: GRPO "สร้าง" หรือแค่ "เหลา"
10. สรุป (Summary)

---

## หัวใจของโน้ตบุ๊กนี้

หัวข้อที่ 7 มีเซลล์ที่สุ่มคำตอบจริงจากโมเดลมาหนึ่งกลุ่ม แล้วคิดเลข advantage
ของสมการ GRPO **ด้วยมือทีละบรรทัด** ก่อน assert ว่าตรงกับสูตรเวกเตอร์ทุกตำแหน่ง
รวมถึงกรณีเสื่อม (degenerate case) ที่ reward เท่ากันทั้งกลุ่ม ซึ่งจะพิมพ์ประโยค
ที่สำคัญที่สุดของบทออกมา: **advantage = 0 → batch นี้ไม่สอนอะไรเลย**

และหัวข้อที่ 8 จะ plot เส้นที่แทบไม่มีใคร plot คือ **สัดส่วนของกลุ่มที่ reward
ยังมีความแตกต่าง (std > 0)** — เส้นแบ่งระหว่าง "โมเดลอิ่มตัว" กับ
"การเรียนรู้หยุดไปนานแล้วโดยที่ GPU ยังร้อนอยู่"

## FAST_MODE — ความซื่อตรงเรื่อง config

| | โจทย์ | G (คำตอบ/โจทย์) | max tokens | optimizer steps | เวลาเทรนโดยประมาณ |
|---|---|---|---|---|---|
| `FAST_MODE = True` (ค่าเริ่มต้น) | 64 | 4 | 192 | 8 | ~8-10 นาที |
| `FAST_MODE = False` (config เต็มที่ใช้รายงานผลในบทความ) | 128 | 8 | 256 | 16 | ~18 นาที |

GRPO เป็น online RL: งบจริงคือ **token ที่ต้อง generate ก่อนทุก step** ไม่ใช่ตัว backprop
(config เต็ม = 128 × 8 × 256 ≈ 262k token, FAST ≈ 49k token)
เราบอกตรง ๆ แบบนี้เพราะบทความที่ไม่บอกว่าตัวเลขสวย ๆ มาจาก config ไหน กำลังโกหกคุณครึ่งประโยค

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16`, `attn_implementation="sdpa"` และ `fp16=True` เอง
  อย่างชัดเจนทุกครั้ง (รายละเอียดอยู่ในคอมเมนต์ของ Cell 0 — อ่านให้ครบ)
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง
- **ห้ามเทรนบน KobEval-TH เด็ดขาด** ชุดนี้เป็นชุดวัดผลอย่างเดียว
  ข้อมูลที่ใช้เทรนในตอนนี้มาจาก `VISAI-AI/gsm8k-thai` (train split)
  ซึ่งแยกขาดจากชุดวัดผล และ held-out สำหรับ pass@k ก็แยกจากชุดเทรนอีกชั้น

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (FAST_MODE, รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลดโมเดล + วัด baseline (TH-MATH 30 ข้อ) | ~3 นาที |
| เตรียมข้อมูล + reward + advantage ด้วยมือ | ~1 นาที |
| pass@k ก่อนเทรน (12 ข้อ × 8 คำตอบ) | ~2 นาที |
| เทรน GRPO (FAST_MODE) | ~8 นาที |
| pass@k หลังเทรน + วัด KobEval-TH ซ้ำ + export | ~5 นาที |
| **รวม** | **~20 นาที** |

ถ้า T4 ที่ได้ช้ากว่าปกติ ให้ลด `N_TRAIN` หรือ `N_HELDOUT` ในหัวข้อที่ 6 ลง


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
if not os.path.exists("/content/thai-llm-tutorials"):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")

---

## 1. ปัญหา (Problem statement)

ลองตั้งโจทย์แบบนี้: สอน Qwen3-0.6B ให้แก้โจทย์คณิตศาสตร์ภาษาไทย
คำตอบสุดท้ายเป็นตัวเลขหนึ่งตัว ตรวจถูกผิดได้ด้วยเครื่องหมาย `==` บรรทัดเดียว

เอาเครื่องมือจากบทก่อน ๆ มาไล่ดูทีละตัว จะพบว่าไม่มีตัวไหนพอดีกับงานนี้เลย:

| วิธี | โมเดลสุ่มคำตอบเองแล้วเรียนจากมันได้ไหม | ต้องมี label จากมนุษย์ | โมเดลใน VRAM |
|---|---|---|---|
| SFT | ไม่ได้ — เลียนแบบเฉลยอย่างเดียว | เฉลยที่คนเขียน | 1 |
| PPO | ได้ | คู่ preference สำหรับเทรน reward model | 4 |
| DPO (ตอนที่ 4) | ไม่ได้ — offline ล้วน | คู่ preference | 2 (LoRA เหลือ 1) |

- **SFT** สอนให้เลียนแบบวิธีทำของเฉลย แต่ไม่เคยให้โมเดลลองผิดลองถูกเอง
- **PPO** ให้โมเดลลองเองได้ แต่แลกด้วย reward model + value network อีกทั้งตัว
  ทั้งที่งานนี้ reward เขียนเป็นฟังก์ชัน Python ได้ตรง ๆ
- **DPO** ตัด RL ทิ้งได้สวยงาม แต่จัดอันดับได้แค่คำตอบที่*มีอยู่แล้ว*ในชุดข้อมูล
  โจทย์เลขต้องการให้โมเดลลองหลาย ๆ ทางแล้วเสริมทางที่ไปถึงคำตอบถูก

คำถามของตอนนี้จึงแคบและคม: **ใน PPO มีชิ้นส่วนไหนที่จำเป็นจริง
และชิ้นไหนลบทิ้งได้ เมื่อ reward ของเราตรวจได้ด้วยโค้ด**

---

## 2. เราจะทำอะไร (Solution)

หน้าที่เดียวของ value network ใน PPO คือตอบว่า *"โดยเฉลี่ยแล้ว prompt นี้ควรได้
reward ประมาณเท่าไหร่"* เพื่อใช้เป็น **baseline** หักออกจาก reward จริง
PPO ตอบคำถามนั้นด้วยการ**เทรนโมเดลอีกตัวทั้งตัว** — GRPO ตอบด้วยการ**สุ่มให้เห็นกับตา**:

> **แนวคิดหลักของตอนนี้:** สุ่มคำตอบ $G$ อันจาก prompt เดียวกัน แล้วเฉลี่ย reward ของกลุ่ม —
> ค่าเฉลี่ยนั้นคือ unbiased estimate ของ "reward ที่คาดหวังจาก prompt นี้" อยู่แล้วโดยนิยาม
> **value network ทั้งตัวจึงลบทิ้งได้** และเมื่อ reward ตรวจด้วยโค้ด (คำตอบเลขถูกหรือผิด)
> reward model กับข้อมูล preference จากมนุษย์ก็หายตามไปด้วย — เหลือศูนย์ label

นี่คือ **GRPO (Group Relative Policy Optimization)** เสนอโดย Shao และคณะ (2024)
ใน DeepSeekMath และเป็นเครื่องยนต์ตัวเดียวกับที่เทรน DeepSeek-R1
แนวทางนี้มีชื่อเรียกรวม ๆ ว่า **RLVR (RL with Verifiable Rewards)**

และขอวางความคาดหวังให้ตรงตั้งแต่ต้น: หลักฐานปัจจุบันชี้ว่า RL แบบนี้ส่วนใหญ่ทำหน้าที่
**"เหลา" ความสามารถที่โมเดลฐานมีอยู่แล้วที่ pass@8 ให้ย้ายมาโผล่ที่ pass@1**
มากกว่าจะสร้างความสามารถใหม่จากศูนย์ — เราจะวัดเรื่องนี้ตรง ๆ ในหัวข้อ 9


---

## 3. สมการ (Equation)

### 3.1 Group-relative advantage — หัวใจทั้งหมดอยู่บรรทัดเดียว

$$
\hat A_i = \frac{r_i - \operatorname{mean}(r_1,\dots,r_G)}{\operatorname{std}(r_1,\dots,r_G)}
$$

- $G$ = จำนวนคำตอบที่สุ่มจาก prompt เดียวกัน (config เต็มคือ 8, FAST_MODE คือ 4)
- $r_i$ = reward ของคำตอบที่ $i$
- **ทุก token ของคำตอบที่ $i$ ใช้ $\hat A_i$ ตัวเดียวกันทั้งประโยค** —
  ต่างจาก PPO ที่พยายามให้ advantage ละเอียดราย token ผ่าน value network และ GAE

อ่านเป็นภาษาคน: **"คำตอบนี้ดีกว่าหรือแย่กว่าความพยายามครั้งอื่น ๆ ของฉันเอง
ต่อโจทย์ข้อเดียวกัน"** — ไม่มีการเปรียบเทียบข้ามโจทย์ ไม่มีการทำนายอนาคต

ผลตามมาที่สำคัญมาก: ถ้าทั้งกลุ่มได้ reward เท่ากันหมด (ถูกหมดหรือผิดหมด)
ทุก $\hat A_i$ เป็นศูนย์ และ batch นั้น**ไม่สอนอะไรเลย** — จำประโยคนี้ไว้
มันจะกลายเป็นทั้งกับดักอันดับหนึ่งและตัวชี้วัดที่สำคัญที่สุดของตอนนี้

### 3.2 GRPO objective ฉบับเต็ม

$$
\mathcal{J}_{\text{GRPO}}(\theta) = \mathbb{E}\left[\frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}\Big\{\min\!\big(\rho_{i,t}\,\hat A_i,\ \operatorname{clip}(\rho_{i,t},\,1-\epsilon,\,1+\epsilon)\,\hat A_i\big) - \beta\,\mathbb{D}_{\text{KL}}\big[\pi_\theta \,\|\, \pi_{\text{ref}}\big]\Big\}\right]
$$

โดย $\rho_{i,t}$ คืออัตราส่วนความน่าจะเป็นของ token เทียบกับ policy ตอนสุ่ม rollout

- $\min(\cdot,\operatorname{clip}(\cdot))$ = PPO clip เดิม ไม่มีอะไรใหม่
- $\frac{1}{|o_i|}$ = เฉลี่ยต่อ token กันคำตอบยาวได้อิทธิพลเกินตัว (length bias จากตอนที่ 4)
- $\beta\,\mathbb{D}_{\text{KL}}$ = สายจูงเส้นเดิมที่ผูกกับ $\pi_{\text{ref}}$

สิ่งที่ควรอ่านคือสิ่งที่**ไม่อยู่**ในสมการ: ไม่มี $V(s)$ ไม่มี GAE ไม่มี critic loss

### 3.3 พจน์ KL ไม่ได้คำนวณตรง ๆ — รู้จัก k3 estimator

$$
\hat{\mathbb{D}}_{k3} = \frac{\pi_{\text{ref}}(o_{i,t})}{\pi_\theta(o_{i,t})} - \log\frac{\pi_{\text{ref}}(o_{i,t})}{\pi_\theta(o_{i,t})} - 1
$$

ทำไมไม่ใช้ $\log(\pi_\theta/\pi_{\text{ref}})$ ตรง ๆ (เรียกว่า k1) ทั้งที่ unbiased เหมือนกัน?
เพราะ k1 ราย sample **ติดลบได้** (ราว 40% ของ sample) ทั้งที่ KL เป็นลบไม่ได้โดยนิยาม
และ variance สูงมาก ให้ $x = \pi_{\text{ref}}/\pi_\theta$ แล้วสังเกต:

1. $x - 1 \geq \log x$ เสมอ ดังนั้น k3 $\geq 0$ **ทุก sample**
2. $\mathbb{E}_{\pi_\theta}[x] = 1$ ดังนั้นพจน์ $(x-1)$ คือ **control variate**
   ที่หักล้าง noise โดยไม่แตะค่าคาดหวัง

รูปในหัวข้อ 4 จะจำลองให้เห็นกับตา (เฉลยจริงรู้ค่าแน่นอน)

### 3.4 หมายเหตุขั้นสูง: การหารด้วย std ไม่ได้บริสุทธิ์อย่างที่เห็น (Dr.GRPO)

กลุ่มที่ reward เกือบเท่ากันหมด (std เล็ก) จะถูก**ขยาย** advantage ด้วยตัวคูณมหาศาล
ขณะที่กลุ่มที่เสียงแตกจริง ๆ (std ใหญ่ — information มากที่สุด) ถูกกดให้เบาลงโดยเปรียบเทียบ
งาน Dr.GRPO (Liu และคณะ, 2025) เสนอให้ตัดการหาร std ทิ้ง เหลือแค่ลบ mean
(ใน TRL 0.16.1 ปุ่มนี้คือ `scale_rewards=False` ใน `GRPOConfig` — เราใช้ค่า default
คือหาร std ตาม GRPO ต้นฉบับ แต่ควรรู้ว่าปุ่มนี้มีอยู่)

### 3.5 pass@k แบบ unbiased — เครื่องมือที่หัวข้อ 9 ต้องใช้

สุ่มคำตอบ $n$ ครั้งต่อโจทย์ ถูก $c$ ครั้ง:

$$
\widehat{\text{pass@}k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}
$$

สูตรที่คนมักใช้ผิดคือ $1-(1-c/n)^k$ ซึ่ง bias เข้าข้างตัวเองอย่างเป็นระบบเมื่อ $n$ เล็ก
เราไม่ต้องเขียนเอง — `kobeval` มี `pass_at_k(n, c, k)` ที่ implement สูตร unbiased
ไว้แล้ว (ตาม Chen และคณะ 2021) และมันคือมาตรวัดที่ใช้ตัดสินว่า GRPO
"สร้าง" ความสามารถใหม่ หรือแค่ "เหลา" ของเดิม


---

## 4. เห็นภาพสมการ (Visualize)

รูปของหัวข้อนี้ถูกวาดในเซลล์**ถัดจาก Cell 1** เพราะกติกาของซีรีส์กำหนดว่า
เซลล์โค้ดที่สองของทุกตอนต้องเป็นการวัด baseline เสมอ ห้ามมีอะไรมาแทรกก่อน

รูปที่จะได้เห็น: (ก) advantage ของกลุ่มตัวอย่าง 8 คำตอบ พร้อมกลุ่มเสื่อมที่ std = 0
และ (ข) การเปรียบเทียบ k1 กับ k3 estimator บนโจทย์สังเคราะห์ที่รู้เฉลย KL จริง

---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 ซึ่งวัดผลโมเดลตั้งต้นด้วย KobEval-TH ก่อนที่เราจะเทรนอะไรทั้งนั้น
สังเกตค่า `th_ratio` เป็นพิเศษ — มันคือสัดส่วนตัวอักษรไทยในคำตอบ
ซึ่งตอนนี้เกี่ยวโดยตรง เพราะ reward หนึ่งในสามตัวของเราจ่ายให้ "คิดเป็นภาษาไทย"

> **หมายเหตุเรื่องเวลา:** เรารัน KobEval-TH เฉพาะ slice `TH-MATH` (30 ข้อ)
> เพราะ GRPO ในตอนนี้เทรนบนโจทย์เลขไทยโดยตรง TH-MATH จึงวัดตรงที่สุด
> เพิ่ม slice อื่นใน `KOBEVAL_SLICES` ได้ถ้ามีเวลา — สัญญาการวัดผลเหมือนกันทุกประการ


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนเทรน/ก่อนแก้อะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
import gc
import json
import math
import re
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
KOBEVAL_SLICES = ["TH-MATH"]
DEV = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)
model.eval()

print(f"โหลดโมเดลแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"vocab_size: {model.config.vocab_size}")

VRAM_AFTER_BASE = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"VRAM ที่ใช้หลังโหลดโมเดลฐาน: {VRAM_AFTER_BASE:.3f} GB")

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
t0 = time.time()
baseline = evaluate(
    model,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-0.6B (baseline)",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัด baseline: {time.time() - t0:.0f} วินาที")

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

print(f"\nรวมทุก slice: acc={baseline['overall']['accuracy']:.3f}  "
      f"th_ratio={baseline['overall']['th_ratio']:.2f}")


---

### รูปประกอบหัวข้อที่ 4 — สองเซลล์ถัดไปไม่ใช้ GPU และไม่ใช้เน็ต

**รูปที่ (ก)** วาดสมการ 3.1 ตรง ๆ บนกลุ่มตัวอย่าง 8 คำตอบภายใต้ reward shaping
ของตอนนี้ (ถูก +1.0, format +0.3, ภาษาไทย +0.2): สังเกตว่าคำตอบที่ได้ reward
เป็นบวก (0.3) ยังโดนผลักลงได้ — เพราะเกณฑ์ไม่ใช่ "ดีไหม" แต่คือ "ดีกว่าเพื่อนร่วมกลุ่มไหม"
แผงขวาคือโหมดตายเงียบ: reward เท่ากันหมด = std เป็นศูนย์ = ไม่มีการเรียนรู้

**รูปที่ (ข)** จำลอง k1 กับ k3 บนโจทย์ที่รู้เฉลย: $\pi_\theta = N(0,1)$,
$\pi_{\text{ref}} = N(0.5,1)$ ทำให้ KL จริง $= 0.125$ พอดี (คำนวณได้ด้วยมือ:
$\mathbb{D}_{KL} = \frac{(\mu_1-\mu_2)^2}{2\sigma^2} = \frac{0.25}{2}$)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 (ก) -- group advantage ของกลุ่มจริง vs กลุ่มเสื่อม (CPU ล้วน)
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

from kobeval import use_thai_font

use_thai_font()

# กลุ่มตัวอย่าง 8 คำตอบ ภายใต้ reward สูงสุด 1.5 (ถูก 1.0 + format 0.3 + ไทย 0.2)
r_example = np.array([1.5, 0.3, 0.5, 1.5, 0.3, 0.0, 0.3, 0.5])
r_degen = np.full(8, 0.3)                       # ทุกคำตอบได้ format reward เท่ากันหมด

def adv(r):
    s = r.std(ddof=1)                           # ddof=1 ตรงกับ torch.std ที่ TRL ใช้
    return (r - r.mean()) / (s + 1e-4)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, r, title in [
    (axes[0], r_example, "กลุ่มจริง: ความต่างภายในกลุ่มคือเชื้อเพลิง"),
    (axes[1], r_degen, "กลุ่มเสื่อม: reward เท่ากันหมด → advantage = 0 ทั้งกลุ่ม"),
]:
    a = adv(r)
    colors = ["#16a34a" if v > 0 else "#dc2626" if v < 0 else "#94a3b8" for v in a]
    ax.bar(range(1, 9), a, color=colors)
    ax.axhline(0, color="#334155", lw=1)
    for i, (ri, ai) in enumerate(zip(r, a), 1):
        ax.text(i, ai + (0.08 if ai >= 0 else -0.16), f"r={ri}", ha="center", fontsize=8)
    ax.set_xlabel("คำตอบที่ i ในกลุ่ม (G = 8)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
axes[0].set_ylabel(r"advantage $\hat{A}_i$")

fig.tight_layout()
fig.savefig("fig_group_advantage.png", dpi=150, bbox_inches="tight")
plt.show()

print("advantage ของกลุ่มจริง :", np.round(adv(r_example), 2))
print("advantage ของกลุ่มเสื่อม:", np.round(adv(r_degen), 2))
print("สังเกต: r=0.3 เป็น reward บวก แต่ advantage ติดลบ เพราะแพ้ค่าเฉลี่ยของกลุ่ม")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 (ข) -- k1 vs k3: unbiased เหมือนกัน ใช้งานได้ไม่เหมือนกัน (CPU ล้วน)
# pi_theta = N(0,1), pi_ref = N(0.5,1)  ->  KL จริง = 0.5^2 / 2 = 0.125 พอดี
# -----------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
N_SAMPLES = 20_000
x = rng.standard_normal(N_SAMPLES)              # สุ่มจาก pi_theta

# log(pi_ref(x) / pi_theta(x)) ของ Gaussian คู่นี้ = 0.5*x - 0.125 (คิดมือได้)
log_ratio_ref_over_theta = 0.5 * x - 0.125

k1 = -log_ratio_ref_over_theta                  # log(pi_theta/pi_ref) -- naive estimator
ratio = np.exp(log_ratio_ref_over_theta)        # x ในสูตร k3 คือ pi_ref/pi_theta
k3 = ratio - log_ratio_ref_over_theta - 1.0

TRUE_KL = 0.125

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
bins = np.linspace(-1.5, 1.5, 80)
axes[0].hist(k1, bins=bins, alpha=0.6, color="#dc2626", label=f"k1 (std={k1.std():.3f})")
axes[0].hist(k3, bins=bins, alpha=0.6, color="#2563eb", label=f"k3 (std={k3.std():.3f})")
axes[0].axvline(0, color="#334155", lw=1, ls=":")
axes[0].set_xlabel("ค่าประมาณ KL ราย sample")
axes[0].set_ylabel("จำนวน sample")
axes[0].set_title("k1 ติดลบได้และกระจายกว้าง / k3 ไม่ติดลบเลย", fontsize=10)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_axisbelow(True)

n_pts = np.arange(1, N_SAMPLES + 1)
axes[1].plot(n_pts, np.cumsum(k1) / n_pts, color="#dc2626", lw=1.2, label="running mean k1")
axes[1].plot(n_pts, np.cumsum(k3) / n_pts, color="#2563eb", lw=1.2, label="running mean k3")
axes[1].axhline(TRUE_KL, color="#16a34a", lw=1.5, ls="--", label=f"KL จริง = {TRUE_KL}")
axes[1].set_xscale("log")
axes[1].set_xlabel("จำนวน sample (log scale)")
axes[1].set_ylabel("ค่าเฉลี่ยสะสม")
axes[1].set_title("ทั้งคู่ลู่เข้าเฉลยเดียวกัน แต่ k3 นิ่งกว่ามากที่ batch เล็ก", fontsize=10)
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)

fig.tight_layout()
fig.savefig("fig_k3_estimator.png", dpi=150, bbox_inches="tight")
plt.show()

frac_neg_k1 = float((k1 < 0).mean())
print(f"KL จริง = {TRUE_KL}")
print(f"mean k1 = {k1.mean():.4f}  (unbiased)   std k1 = {k1.std():.4f}   "
      f"ติดลบ {100 * frac_neg_k1:.1f}% ของ sample")
print(f"mean k3 = {k3.mean():.4f}  (unbiased)   std k3 = {k3.std():.4f}   "
      f"ติดลบ {100 * float((k3 < 0).mean()):.1f}% ของ sample")
assert (k3 >= 0).all(), "k3 ต้องไม่ติดลบเลยแม้แต่ sample เดียว (x-1 >= log x เสมอ)"
print("=> k3 unbiased เท่ากัน แต่ variance ต่ำกว่าและไม่ติดลบ -- ใช้เป็น penalty ได้ตั้งแต่ step แรก")


---

### 5.1 policy = LoRA, reference = ฐานที่ปิด adapter — สูตรเดิมจากตอนที่ 4

GRPO ต้องใช้ทั้ง $\pi_\theta$ และ $\pi_{\text{ref}}$ — ตอนที่ 4 สอนเราแล้วว่า
ถ้าเทรนด้วย **LoRA**, TRL จะไม่โหลด reference แยก: มันใช้โมเดลฐานตัวเดิม
ที่**ปิด adapter** เป็น reference ให้เอง ต้นทุน VRAM ของ reference = **ศูนย์ไบต์**
ต้นทุนโมเดลสุทธิของ GRPO จึงเท่ากับ SFT ธรรมดา (นี่คือสิ่งที่รูป 5.2 ในบทความชี้)

### ต่อยอดจากตอนที่ 2 ถ้ามี adapter อยู่ — ไม่มีก็เริ่มใหม่ได้

ในลำดับของซีรีส์ ตอนที่ 2 เซฟ LoRA adapter ที่ผ่าน SFT ไว้ที่ `qwen3-0.6b-th-sft-lora/`
ถ้าคุณรันตอนที่ 2 ใน session เดียวกัน (หรืออัปโหลด adapter ขึ้นมา) เซลล์ถัดไปจะ
**โหลดมาต่อยอดให้อัตโนมัติ** ถ้าไม่มี ก็สร้าง adapter ใหม่จาก `Qwen/Qwen3-0.6B` ตรง ๆ
— โน้ตบุ๊กนี้ตั้งใจให้**รันจบได้ด้วยตัวเอง** ไม่อ้างอิงไฟล์ภายนอกที่ยังไม่มีอยู่จริง

### กับดัก fp16 ที่ต้องปิดก่อนเริ่ม (เหมือนตอนที่ 4 ทุกตัวอักษร)

`fp16=True` แปลว่า *mixed precision* — ถ้าพารามิเตอร์ที่เทรนได้ (LoRA adapter)
เป็น fp16 `GradScaler` จะโยน `ValueError: Attempting to unscale FP16 gradients.`
วิธีแก้คือ cast **เฉพาะพารามิเตอร์ที่เทรนได้** เป็น fp32 ส่วนน้ำหนักฐานยังเป็น fp16 เหมือนเดิม


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 5.1 -- ติด LoRA: โหลด adapter จากตอนที่ 2 ถ้ามี ไม่มีก็สร้างใหม่
# -----------------------------------------------------------------------------
from peft import LoraConfig, PeftModel, get_peft_model

ADAPTER_CANDIDATES = ["qwen3-0.6b-th-sft-lora", "/content/qwen3-0.6b-th-sft-lora"]
adapter_path = next((p for p in ADAPTER_CANDIDATES if os.path.isdir(p)), None)

if adapter_path is not None:
    policy = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)
    ADAPTER_SOURCE = f"โหลดต่อยอดจากตอนที่ 2: {adapter_path}"
else:
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    policy = get_peft_model(model, lora_cfg)
    ADAPTER_SOURCE = "สร้างใหม่ (ไม่พบ adapter ของตอนที่ 2 -- ปกติสำหรับ Colab เปิดใหม่)"

print(f"adapter: {ADAPTER_SOURCE}")


# cast เฉพาะพารามิเตอร์ที่เทรนได้เป็น fp32 (ดูเหตุผลด้านบน)
def cast_trainable_to_fp32(m):
    n_cast = 0
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
            n_cast += 1
    return n_cast


n_cast = cast_trainable_to_fp32(policy)

VRAM_AFTER_LORA = torch.cuda.memory_allocated() / (1024 ** 3)
n_train = sum(p.numel() for p in policy.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in policy.parameters())

print(f"พารามิเตอร์ที่เทรน : {n_train / 1e6:.2f}M จาก {n_total / 1e6:.1f}M "
      f"({100 * n_train / n_total:.2f}%)")
print(f"cast เป็น fp32 ไป  : {n_cast} tensors")
print(f"VRAM ก่อนติด adapter : {VRAM_AFTER_BASE:.3f} GB")
print(f"VRAM หลังติด adapter : {VRAM_AFTER_LORA:.3f} GB")
print(f"ส่วนต่าง             : {VRAM_AFTER_LORA - VRAM_AFTER_BASE:.3f} GB")
print()
print("reference model ของ GRPO ไม่กิน VRAM เพิ่มเลยแม้แต่ไบต์เดียว --")
print("TRL เห็นว่า policy เป็น PeftModel แล้วใช้ 'ฐานที่ปิด adapter' เป็น reference ให้เอง")


---

## 6. เตรียมข้อมูล (Data)

เราใช้ **`VISAI-AI/gsm8k-thai`** — ชุดโจทย์คณิตศาสตร์ GSM8K ฉบับแปลไทย
สุ่มโจทย์จาก train split และแบ่ง **held-out ไว้วัด pass@k ต่างหาก ไม่แตะระหว่างเทรน**

สังเกตว่า**ไม่มีคอลัมน์ chosen/rejected และไม่มี label มนุษย์ใด ๆ** —
มีแค่โจทย์กับเฉลยตัวเลข สิ่งที่แทน label คือ reward สามฟังก์ชันที่ตรวจด้วยโค้ดล้วน ๆ:

| reward | เงื่อนไข | ค่า |
|---|---|---|
| `reward_correct` | ตัวเลขสุดท้ายหลัง `</think>` ตรงเฉลย (แปลงเลขไทย ๐-๙ ก่อน) | +1.0 |
| `reward_format` | มี `<think>...</think>` ที่**ไม่ว่างเปล่า** (`.+?` ไม่ใช่ `.*?`) | +0.3 |
| `reward_thai` | `th_ratio` ของคำตอบ > 0.5 (ใช้ตัววัดกลางจาก `kobeval`) | +0.2 |

reward รวมของคำตอบหนึ่งอันคือผลบวกของทั้งสาม: สูงสุด 1.5 ต่ำสุด 0.0

> **ทำไมต้องมี reward ย่อย ไม่ใช่แค่ "ถูก/ผิด"** — ช่วงต้นของการเทรน โมเดล 0.6B
> ตอบโจทย์เลขถูกน้อยมาก ถ้า reward มีแค่ถูก/ผิด กลุ่มส่วนใหญ่จะเป็น [0,0,0,0]
> คือ std ศูนย์ gradient ศูนย์ **เทรนฟรีไม่ได้อะไร** (รูปที่ (ก) แผงขวา)
> reward ย่อยเรื่อง format กับภาษาไทยทำให้กลุ่มยังมีความต่างให้เรียน
> ตั้งแต่ก่อนโมเดลจะเริ่มตอบถูก — นี่คือ reward shaping ตรงตัวที่สุด
> และหมายเหตุตัวโต ๆ: **มันเปิดช่องโกงด้วย**

> **บทเรียน reward hacking ที่เกิดขึ้นจริง:** เวอร์ชันแรกของ `reward_format`
> ใช้ regex `<think>.*?</think>` (จุดสำคัญ: `.*?` ยอมรับสตริงว่าง)
> โมเดลค้นพบภายในไม่กี่ step ว่าพิมพ์ `<think></think>` เปล่า ๆ แล้วเดาเลขมั่ว
> เก็บ +0.3 ได้ฟรี ถูกกว่าการคิดจริงมาก — mean reward ไต่สวยแต่ accuracy ไม่ขยับ
> เราแก้เป็น `.+?` บังคับให้ต้องมีเนื้อหา และโน้ตบุ๊กนี้**ดักจับความพยายามโกงแบบเดิม**
> ระหว่างเทรนไว้ให้ดูในหัวข้อ 8 ด้วย (`HACK_ATTEMPTS`)
> บทเรียน: **โมเดลไม่ได้ optimize สิ่งที่คุณตั้งใจ มัน optimize สิ่งที่คุณเขียน**

หมายเหตุเรื่อง prompt: เราสร้าง prompt ด้วย chat template ของ Qwen3 แบบ
`enable_thinking=True` เพราะ reward ต้องการให้โมเดล**เขียน** `<think>...</think>` เอง
(ต่างจากสัญญาการวัดผลของ `kobeval` ที่ปิด thinking — นั่นคือฝั่งวัดผล ไม่ใช่ฝั่งเทรน)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.1 -- โหลด VISAI-AI/gsm8k-thai แล้วประกอบชุดเทรน + held-out
# -----------------------------------------------------------------------------
from datasets import Dataset, load_dataset

from kobeval import extract_final_int, thai_digits_to_arabic

FAST_MODE = True                     # False = config เต็มที่ใช้รายงานผลในบทความ

N_TRAIN = 64 if FAST_MODE else 128   # จำนวนโจทย์ที่ใช้เทรน
G = 4 if FAST_MODE else 8            # ขนาดกลุ่ม (num_generations)
MAX_COMPLETION = 192 if FAST_MODE else 256
N_HELDOUT = 12                       # โจทย์ held-out สำหรับ pass@k (ไม่แตะระหว่างเทรน)
PASSK_N = 8                          # สุ่ม n คำตอบต่อโจทย์ตอนวัด pass@k

SYSTEM = "จงคิดทีละขั้นใน <think>...</think> แล้วจบด้วยคำตอบเป็นตัวเลขบรรทัดสุดท้าย"


def pick_column(columns, candidates):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


def to_prompt(question):
    """prompt ตาม chat template ของ Qwen3 -- เปิด thinking เพราะ reward ต้องการ <think>"""
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )


def gold_int(answer_text):
    """เฉลย GSM8K อยู่ท้ายสุด (มักตามหลัง '####') -- แปลงเลขไทยก่อน แล้วดึงจำนวนเต็มตัวสุดท้าย"""
    text = thai_digits_to_arabic(str(answer_text))
    if "####" in text:
        text = text.split("####")[-1]
    return extract_final_int(text)     # ตัวดึงเลขมาตรฐานของ kobeval (คืน None ถ้าไม่ใช่จำนวนเต็ม)


raw = load_dataset("VISAI-AI/gsm8k-thai", split="train")
print(f"VISAI-AI/gsm8k-thai columns: {raw.column_names}  n={len(raw)}")

q_col = pick_column(raw.column_names, ["question", "question_th", "instruction", "problem", "input"])
a_col = pick_column(raw.column_names, ["answer", "answer_th", "solution", "output"])
assert None not in (q_col, a_col), f"schema ไม่ตรงที่คาด: {raw.column_names}"

raw = raw.shuffle(seed=SEED)
items = []
for row in raw:
    q, a = row[q_col], row[a_col]
    if not (isinstance(q, str) and 20 <= len(q.strip()) <= 300):
        continue
    gold = gold_int(a)
    if gold is None:
        continue
    items.append({"prompt": to_prompt(q.strip()), "question": q.strip(), "answer": int(gold)})
    if len(items) >= N_TRAIN + N_HELDOUT:
        break

train_items = items[:N_TRAIN]
heldout_items = items[N_TRAIN:N_TRAIN + N_HELDOUT]

# GRPOTrainer ต้องการคอลัมน์ "prompt" ส่วนคอลัมน์อื่น (answer) จะถูกส่งเข้า reward function
train_ds = Dataset.from_list([{"prompt": it["prompt"], "answer": it["answer"]} for it in train_items])

print(f"โจทย์เทรน {len(train_items)} ข้อ | held-out {len(heldout_items)} ข้อ (แยกขาดจากกัน)")
print(f"FAST_MODE={FAST_MODE}  ->  G={G}, max_completion={MAX_COMPLETION}")
print()
print("ตัวอย่างโจทย์ข้อแรก:")
print("  คำถาม :", train_items[0]["question"][:120].replace(chr(10), " "))
print("  เฉลย  :", train_items[0]["answer"])
print()
print("ท้าย prompt หลังผ่าน chat template (สังเกตว่าจบที่ assistant พร้อมให้คิดเอง):")
print(repr(train_items[0]["prompt"][-90:]))


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.2 -- reward สามฟังก์ชัน + ตรวจสอบด้วยตัวอย่างที่รู้เฉลย (เซลล์นี้ไม่ใช้ GPU)
# -----------------------------------------------------------------------------
THINK_RE = re.compile(r"<think>.+?</think>", re.DOTALL)   # .+? ไม่ใช่ .*? -- ดูบทเรียนโกงด้านบน
EMPTY_THINK_RE = re.compile(r"<think>\s*</think>")


def grpo_extract_final_int(text):
    """ดึงจำนวนเต็มตัวสุดท้าย 'หลัง </think>' -- แปลงเลขไทย ๐-๙ เป็นอารบิกก่อนเสมอ"""
    text = thai_digits_to_arabic(text)
    tail = text.split("</think>")[-1]         # ตรวจเฉพาะส่วนหลังการคิด
    nums = re.findall(r"-?\d[\d,]*", tail)
    if not nums:
        return None
    try:
        return int(nums[-1].replace(",", ""))
    except ValueError:
        return None


def reward_correct(completions, answer, **kwargs):
    """+1.0 ถ้าตัวเลขสุดท้ายตรงเฉลย"""
    return [1.0 if grpo_extract_final_int(c) == a else 0.0
            for c, a in zip(completions, answer)]


def reward_format(completions, **kwargs):
    """+0.3 ถ้ามี <think>...</think> ที่ไม่ว่างเปล่า"""
    return [0.3 if THINK_RE.search(c) else 0.0 for c in completions]


def reward_thai(completions, **kwargs):
    """+0.2 ถ้า th_ratio > 0.5 -- ใช้ตัววัดกลางของ kobeval ตัวเดียวกับทั้งซีรีส์"""
    return [0.2 if th_ratio(c) > 0.5 else 0.0 for c in completions]


# --- ตรวจ reward ด้วยตัวอย่างที่รู้เฉลย ก่อนปล่อยให้มันตัดสิน gradient จริง ---------
assert thai_digits_to_arabic("๑๒๓") == "123"
assert thai_digits_to_arabic("คำตอบคือ ๔๒ บาท") == "คำตอบคือ 42 บาท"
print("ผ่าน: แปลงเลขไทย ๑๒๓ -> 123, ๔๒ -> 42")

assert grpo_extract_final_int("<think>๒+๒=๔</think> ดังนั้นคำตอบคือ ๔") == 4
assert grpo_extract_final_int("<think>คิด 5*3=15</think>" + chr(10) + "คำตอบคือ 15 บาท") == 15
assert grpo_extract_final_int("<think>1+1=2</think> ไม่มีตัวเลขนอก think") is None  # เลขใน think ไม่นับ
assert grpo_extract_final_int("ตอบ 1,250") == 1250
assert grpo_extract_final_int("ไม่มีตัวเลขเลย") is None
print("ผ่าน: ดึงเลขสุดท้ายหลัง </think> ถูกต้อง (รวมเลขไทยและเลขมีจุลภาค)")

assert reward_correct(["<think>คิด</think> ตอบ ๔๒"], [42]) == [1.0]
assert reward_correct(["<think>คิด</think> ตอบ 41"], [42]) == [0.0]
assert reward_format(["<think>คิดจริง</think> 42"]) == [0.3]
assert reward_format(["<think></think> 42"]) == [0.0], "think ว่างต้องไม่ได้คะแนน (กันฟาร์ม +0.3)"
assert reward_thai(["<think>ลองคิดดู สองบวกสอง</think> คำตอบคือ 4"]) == [0.2]
assert reward_thai(["<think>let me think, 2+2</think> the answer is 4"]) == [0.0]
print("ผ่าน: reward ทั้งสามตัวให้คะแนนตรงตามที่ตั้งใจ รวมทั้งกรณี think ว่างเปล่า")

# --- tracker: บันทึก reward ทุกครั้งที่ TRL เรียก เพื่อ plot สัดส่วนกลุ่มที่ std > 0 ----
TRACK = {"correct": [], "format": [], "thai": [], "lens": []}
HACK_ATTEMPTS = []          # เก็บหลักฐาน <think></think> ว่างเปล่า ถ้าโมเดลลองโกง


def tracked(name, fn, capture=False):
    def wrapper(completions, **kwargs):
        rs = fn(completions=completions, **kwargs)
        TRACK[name].append(list(rs))
        if capture:
            TRACK["lens"].append(
                [len(tokenizer(c, add_special_tokens=False).input_ids) for c in completions]
            )
            for c in completions:
                if EMPTY_THINK_RE.search(c) and len(HACK_ATTEMPTS) < 5:
                    HACK_ATTEMPTS.append(c)
        return rs
    wrapper.__name__ = fn.__name__      # TRL ใช้ชื่อนี้ตอน log
    return wrapper


tr_correct = tracked("correct", reward_correct)
tr_format = tracked("format", reward_format, capture=True)
tr_thai = tracked("thai", reward_thai)
print()
print("ติดตั้ง tracker แล้ว -- ทุก reward ที่ TRL เรียกจะถูกบันทึกไว้วิเคราะห์ในหัวข้อ 8")


---

## 7. โค้ดหลัก (Main code)

### 7.1 GRPOTrainer — คราวนี้ trainer เป็นพลเมืองชั้นหนึ่ง

`GRPOTrainer` คือ trainer ที่ TRL ดูแลเป็นตัวชูโรงหลังกระแส DeepSeek-R1
รับ reward เป็น**ฟังก์ชัน Python ธรรมดา**ตรง ๆ ไม่ต้องห่อโมเดลอะไรทั้งนั้น

คิดเลขงบให้ดูตรง ๆ (config เต็ม): 128 โจทย์ × 8 คำตอบ = 1,024 completion
หารด้วย effective batch 64 completion ต่อ step = **16 optimizer step** — แค่นั้นจริง ๆ
(FAST_MODE: 64 × 4 = 256 completion / 32 ต่อ step = **8 step**)
เวลาที่เหลือเกือบทั้งหมดคือการ generate ก่อนแต่ละ step ไม่ใช่ backprop

> **ตัวเลขสองตัวที่ต้องขยับพร้อมกันเสมอ:** `per_device_train_batch_size` นับเป็น
> **completion ไม่ใช่ prompt** และ (คูณจำนวน process แล้ว) ต้องหารด้วย `num_generations`
> ลงตัว เพราะสมาชิกกลุ่มเดียวกันต้องอยู่ใน batch เดียวกันถึงจะคำนวณ mean/std ของกลุ่มได้
> ถ้าตั้งไม่ลงตัว TRL จะ error ตั้งแต่สร้าง trainer — ซึ่งดีแล้ว พังดังดีกว่าพังเงียบ

> **learning rate ของ online RL ต้องต่ำที่สุดในซีรีส์:** SFT `2e-4` → DPO `5e-6` → GRPO `1e-6`
> เหตุผล: ข้อมูลเทรนของ GRPO คือคำตอบที่โมเดล*ตัวปัจจุบัน*สุ่มออกมา
> ถ้าน้ำหนักขยับแรงจนภาษาเริ่มเพี้ยน คำตอบรุ่นถัดไปจะเพี้ยนตาม แล้ว reward
> จะพังทั้งกระดาน — ความผิดพลาดของ online RL **ทบต้น**
> ไม่เหมือน supervised ที่ข้อมูลเทรนไม่หนีไปไหน

ค่า `temperature=1.0` ห้ามลด — ความหลากหลายในกลุ่มคือเชื้อเพลิงของการเรียนรู้ทั้งระบบ
(ลดต่ำเมื่อไหร่ คำตอบทั้งกลุ่มแทบเหมือนกัน → reward เท่ากัน → std = 0 → ไม่เรียน)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.1 -- ประกอบ GRPOConfig + GRPOTrainer (ยังไม่เทรน -- เทรนจริงในหัวข้อ 8)
# -----------------------------------------------------------------------------
from trl import GRPOConfig, GRPOTrainer

BETA_KL = 0.04       # น้ำหนัก KL penalty (TRL คำนวณด้วย k3 estimator จากหัวข้อ 3.3)
EPSILON = 0.2        # ช่วง clip เดียวกับ PPO
LR = 1e-6            # ต่ำที่สุดในซีรีส์ -- ดูคำเตือนด้านบน
BATCH_COMPLETIONS = 16
GRAD_ACCUM = 2 if FAST_MODE else 4

cfg_grpo = GRPOConfig(
    output_dir="grpo-out",
    num_generations=G,                       # ขนาดกลุ่ม -- FAST: 4, เต็ม: 8
    max_completion_length=MAX_COMPLETION,    # FAST: 192, เต็ม: 256
    max_prompt_length=256,
    temperature=1.0,                         # ห้ามลด -- ความหลากหลายคือเชื้อเพลิง
    beta=BETA_KL,
    epsilon=EPSILON,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_COMPLETIONS,   # นับเป็น completion ต้องหารด้วย G ลงตัว
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    fp16=USE_FP16,                           # บน T4 คือ True
    bf16=SUPPORTS_BF16,                      # บน T4 คือ False
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = GRPOTrainer(
    model=policy,                            # PeftModel -> TRL ใช้ฐานปิด adapter เป็น ref ฟรี
    reward_funcs=[tr_correct, tr_format, tr_thai],
    args=cfg_grpo,
    train_dataset=train_ds,
)

total_completions = N_TRAIN * G
per_step = BATCH_COMPLETIONS * GRAD_ACCUM
print("งบของรันนี้ (คิดเลขให้ดูตรง ๆ):")
print(f"  completion ทั้งหมด        : {N_TRAIN} โจทย์ x {G} = {total_completions}")
print(f"  completion ต่อ optimizer step: {BATCH_COMPLETIONS} x accum {GRAD_ACCUM} = {per_step}")
print(f"  optimizer steps            : {total_completions // per_step}")
print(f"  token generate สูงสุด      : {total_completions * MAX_COMPLETION:,} token")
print()
print("เวลาบน T4 จะหมดไปกับบรรทัดสุดท้ายนั่นแหละ ไม่ใช่ backprop")


### 7.2 คิด advantage เองด้วยมือ — เลขทั้งหมดอยู่ตรงนี้

เพื่อไม่ให้ `GRPOTrainer` เป็นกล่องดำ เซลล์ถัดไปทำสามอย่าง:

1. **สุ่มกลุ่มคำตอบจริง** $G$ อันจากโจทย์เทรนข้อแรก ด้วย temperature 1.0
   (เงื่อนไขเดียวกับที่ trainer จะใช้) แล้วให้ reward ทั้งสามฟังก์ชันตัดสิน
2. คิด $\hat A_i = (r_i - \text{mean})/(\text{std} + \epsilon)$ **ด้วยมือทีละบรรทัด**
   (ผลรวม → mean → ผลต่างยกกำลังสอง → std แบบ $n-1$ ตัวเดียวกับ `torch.std`
   ที่ TRL ใช้) แล้ว `assert` ว่าตรงกับสูตรเวกเตอร์ทุกตำแหน่ง
3. ทดสอบ**กรณีเสื่อม**: reward เท่ากันทั้งกลุ่ม → std = 0 → advantage เป็นศูนย์ทุกตัว
   ซึ่งคือประโยคสำคัญที่สุดของตอนนี้ในรูปของโค้ดที่รันได้

หมายเหตุ: TRL 0.16.1 ใช้ `(r - mean) / (std + 1e-4)` เป๊ะ ๆ (ดู `grpo_trainer.py`)
เราใช้ $\epsilon$ ตัวเดียวกันเพื่อให้เทียบกันได้ถึงทศนิยมตัวสุดท้าย


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 -- group advantage ด้วยมือ บนกลุ่มคำตอบจริงที่โมเดลเพิ่งสุ่มออกมา
# -----------------------------------------------------------------------------
EPS_ADV = 1e-4       # ค่าเดียวกับใน TRL: advantages = (r - mean) / (std + 1e-4)


def group_advantages(rewards, G, eps=EPS_ADV):
    """สมการ 3.1 ในรูปเวกเตอร์: rewards [B] เรียงเป็นกลุ่มละ G ตัวจาก prompt เดียวกัน"""
    r = rewards.view(-1, G)                      # [B/G, G]
    mean = r.mean(dim=1, keepdim=True)
    std = r.std(dim=1, keepdim=True)             # unbiased (n-1) -- ค่า default ของ torch และ TRL
    return ((r - mean) / (std + eps)).view(-1)   # Dr.GRPO: ตัด "/ (std + eps)" ทิ้ง


# --- 1) สุ่มกลุ่มจริงจากโจทย์เทรนข้อแรก --------------------------------------
policy.eval()
demo = train_items[0]
enc = tokenizer(demo["prompt"], return_tensors="pt").to(DEV)
torch.manual_seed(SEED)
with torch.no_grad():
    out = policy.generate(
        **enc,
        do_sample=True, temperature=1.0, top_p=1.0,     # เงื่อนไขเดียวกับตอนเทรน
        num_return_sequences=G,
        max_new_tokens=MAX_COMPLETION,
        pad_token_id=tokenizer.pad_token_id,
    )
group = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

r_total = [a + b + c for a, b, c in zip(
    reward_correct(group, [demo["answer"]] * G),
    reward_format(group),
    reward_thai(group),
)]
r = torch.tensor(r_total, dtype=torch.float32)

# --- 2) คิดด้วยมือทีละบรรทัด แล้ว assert ว่าตรงกับสูตรเวกเตอร์ -----------------
mean_hand = sum(r_total) / G
var_hand = sum((v - mean_hand) ** 2 for v in r_total) / (G - 1)   # n-1 ตรงกับ torch.std
std_hand = var_hand ** 0.5
adv_hand = [(v - mean_hand) / (std_hand + EPS_ADV) for v in r_total]
adv_formula = group_advantages(r, G)

print(f"โจทย์: {demo['question'][:90]}...  (เฉลย = {demo['answer']})")
print(f"{'i':>2s} {'reward':>7s} {'advantage':>10s}   คำตอบ (ตัดมา 60 ตัวอักษร)")
for i, (ri, ai, text) in enumerate(zip(r_total, adv_formula.tolist(), group), 1):
    print(f"{i:2d} {ri:7.2f} {ai:+10.4f}   {text[:60].replace(chr(10), ' ')}")
print(f"mean = {mean_hand:.4f}   std (n-1) = {std_hand:.4f}")

assert torch.allclose(adv_formula, torch.tensor(adv_hand), atol=1e-6), \
    "เลขที่คิดด้วยมือไม่ตรงกับสูตรเวกเตอร์ -- ไม่ควรเกิดขึ้น"
print()
print("assert ผ่าน: advantage ที่คิดด้วยมือ == สูตรเวกเตอร์ ทุกตำแหน่ง (atol=1e-6)")

# --- 3) กรณีเสื่อม: reward เท่ากันทั้งกลุ่ม -----------------------------------
r_flat = torch.full((G,), 0.3)                   # เช่น ทุกคำตอบได้แค่ format reward
adv_flat = group_advantages(r_flat, G)
adv_flat_hand = [(0.3 - 0.3) / (0.0 + EPS_ADV)] * G   # เลขแม่นยำอนันต์: ศูนย์เป๊ะ

# รายละเอียดที่คนมักไม่รู้: ใน fp32 ค่าเฉลี่ยของ 0.3 แปดตัวอาจเหลือเศษ ~1e-8
# (ลำดับการบวกของ mean ไม่ตรงกับตัวเลขเป๊ะ ๆ) พอถูกหารด้วย eps=1e-4 จะโผล่เป็น
# advantage ~1e-4 -- เล็กกว่า advantage จริง (สเกล ~1) ราวหมื่นเท่า = ศูนย์ในทางปฏิบัติ
# TRL ก็ได้เศษแบบเดียวกันนี้ เพราะคำนวณด้วยสูตรเดียวกันเป๊ะ
assert torch.allclose(adv_flat, torch.tensor(adv_flat_hand), atol=1e-3), "กรณีเสื่อมต้องเป็นศูนย์ (ในทางปฏิบัติ)"
assert adv_flat.abs().max().item() < 1e-3
print()
print(f"กรณีเสื่อม r = {r_flat.tolist()}")
print(f"advantage = {adv_flat.tolist()}")
print(f"(เศษ fp32 ที่ใหญ่ที่สุด = {adv_flat.abs().max().item():.2e} -- ศูนย์ในทางปฏิบัติ)")
print("advantage = 0 → batch นี้ไม่สอนอะไรเลย (gradient เป็นศูนย์ทั้งกลุ่ม)")
print("ทุกกลุ่มที่ถูกหมด/ผิดหมด/ได้ reward เท่ากันหมด คือ compute ที่เผาทิ้งเงียบ ๆ แบบนี้")


### 7.3 วัด pass@k **ก่อน**เทรน — เพื่อให้หัวข้อ 9 มีคู่เทียบที่ยุติธรรม

ต้องวัด**ตอนนี้** ก่อน `trainer.train()` เพราะนี่คือ policy ตัวที่กำลังจะถูกเทรน
(ฐาน + adapter ปัจจุบัน) — ถ้าไปวัดทีหลังด้วยการปิด adapter จะได้ "โมเดลฐาน"
ซึ่งไม่ใช่จุดตั้งต้นจริงในกรณีที่โหลด adapter จากตอนที่ 2 มาต่อยอด

วิธีวัด: สุ่ม $n = 8$ คำตอบต่อโจทย์ (temperature 1.0) บน held-out 12 ข้อ
นับ $c$ = จำนวนที่ถูก แล้วใช้ `pass_at_k(n, c, k)` ของ `kobeval` (สูตร unbiased
จากหัวข้อ 3.5) — pass@1 รายงานพร้อม **Wilson 95% CI** จากจำนวนตัวอย่างรวมทั้งหมด


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.3 -- pass@k ก่อนเทรน (held-out 12 ข้อ x 8 คำตอบ, temperature 1.0)
# -----------------------------------------------------------------------------
from kobeval import pass_at_k


def sample_solutions(problems, n=PASSK_N, max_new=None, tag=""):
    """สุ่ม n คำตอบต่อโจทย์จาก policy ปัจจุบัน (adapter เปิด) -- seed แยกต่อข้อ ทำซ้ำได้"""
    max_new = max_new or MAX_COMPLETION
    outs = []
    policy.eval()
    t0 = time.time()
    with torch.no_grad():
        for k, it in enumerate(problems):
            enc = tokenizer(it["prompt"], return_tensors="pt").to(DEV)
            torch.manual_seed(SEED + k)          # ผลของข้อหนึ่งไม่ขึ้นกับจำนวนข้อก่อนหน้า
            out = policy.generate(
                **enc,
                do_sample=True, temperature=1.0, top_p=1.0,
                num_return_sequences=n,
                max_new_tokens=max_new,
                pad_token_id=tokenizer.pad_token_id,
            )
            outs.append(tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                               skip_special_tokens=True))
            if (k + 1) % 4 == 0:
                print(f"  {tag}: {k + 1}/{len(problems)} ข้อ ({time.time() - t0:.0f}s)")
    return outs


def passk_report(problems, sampled, n=PASSK_N):
    per = []
    for it, texts in zip(problems, sampled):
        c = sum(1 for t in texts if grpo_extract_final_int(t) == it["answer"])
        per.append({"answer": it["answer"], "c": c,
                    "pass1": pass_at_k(n, c, 1), "passn": pass_at_k(n, c, n)})
    pass1 = sum(p["pass1"] for p in per) / len(per)
    passn = sum(p["passn"] for p in per) / len(per)
    total_c = sum(p["c"] for p in per)
    lo, hi = wilson_ci(total_c, n * len(per))
    return {"n": n, "k_max": n, "per_problem": per, "pass1": pass1, "passn": passn,
            "wilson_pass1": [lo, hi], "total_correct": total_c,
            "total_samples": n * len(per)}


sampled_before = sample_solutions(heldout_items, tag="pass@k ก่อนเทรน")
PASSK_BEFORE = passk_report(heldout_items, sampled_before)

print()
print(f"ก่อนเทรน: pass@1 = {PASSK_BEFORE['pass1']:.3f} "
      f"[Wilson 95% CI {PASSK_BEFORE['wilson_pass1'][0]:.3f}-{PASSK_BEFORE['wilson_pass1'][1]:.3f}]")
print(f"          pass@{PASSK_N} = {PASSK_BEFORE['passn']:.3f}")
print(f"          ช่องว่าง pass@{PASSK_N} - pass@1 = "
      f"{PASSK_BEFORE['passn'] - PASSK_BEFORE['pass1']:.3f}"
      f"   <- นี่คือ 'ความสามารถที่กระจัดกระจาย' ที่ GRPO จะพยายามย้ายมาที่ pass@1")

# เก็บคำตอบ greedy ของ policy ก่อนเทรนไว้ 4 ข้อ สำหรับ samples.json ในหัวข้อ 9
SAMPLE_PROBLEMS = heldout_items[:4]
sample_before_texts = []
with torch.no_grad():
    for it in SAMPLE_PROBLEMS:
        enc = tokenizer(it["prompt"], return_tensors="pt").to(DEV)
        torch.manual_seed(SEED)
        out = policy.generate(**enc, do_sample=False, max_new_tokens=MAX_COMPLETION,
                              pad_token_id=tokenizer.pad_token_id)
        sample_before_texts.append(
            tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"\nเก็บคำตอบ greedy ก่อนเทรนไว้ {len(sample_before_texts)} ข้อ สำหรับเทียบก่อน-หลัง")


---

## 8. ผลลัพธ์ (Results)

รันเทรนจริง แล้ววัด 4 อย่าง:

1. **Mean reward ต่อรอบ generate** — ควรไต่ขึ้น (นี่คือสิ่งที่ optimizer เห็น)
2. **สัดส่วนของกลุ่มที่ std ไม่เป็นศูนย์** — ตัวชี้วัดที่แทบไม่มีใคร plot
3. **ความยาว completion เฉลี่ย** — เทียบคู่กับ reward (โมเมนต์ mini-R1)
4. **ความพยายามโกง format** — `<think></think>` ว่างเปล่า ที่ดักไว้ระหว่างเทรน

> **ตัวชี้วัดข้อ 2 คือสัญญาณชีพของ GRPO:** mean reward ที่นิ่ง ๆ อ่านได้สองแบบ —
> "โมเดลอิ่มตัวแล้ว" หรือ "การเรียนรู้หยุดไปนานแล้ว" ตัวแยกสองกรณีนี้คือสัดส่วนกลุ่ม
> ที่ std ไม่เป็นศูนย์ **ถ้ามันแตะศูนย์เมื่อไหร่ ทุก batch หลังจากนั้นคือ no-op เงียบ ๆ**:
> loss ยังพิมพ์ออกมา step ยังเดิน GPU ยังร้อน แต่ gradient เป็นศูนย์เป๊ะทุก step
> เทรนต่ออีกชั่วโมงก็ได้ผลเท่าเดิม — นี่ควรเป็นนิสัยของคุณในทุกโปรเจกต์ RLVR

**โมเมนต์ mini-R1 ที่สัญญาไว้:** กระดาษ DeepSeek-R1 มีกราฟโด่งดัง —
ความยาวคำตอบโตขึ้นเอง*พร้อมกับ*ความแม่น โดยไม่มีใครสั่งให้คิดยาว
เรา plot คู่เดียวกันนี้ที่สเกลจิ๋ว ถ้าเห็นทั้งสองเส้นขยับขึ้นด้วยกันแม้เพียงเล็กน้อย
นั่นคือกลไกเดียวกับ R1 ในหลอดทดลองของคุณเอง — และถ้าความยาวโตแต่ reward
ฝั่ง correct ไม่ขยับ ให้สงสัย reward hacking ก่อนเสมอ


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.1 -- เทรนจริง
# -----------------------------------------------------------------------------
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
TRAIN_TIME_S = time.time() - t0
VRAM_PEAK_GB = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"\nเวลาเทรน  : {TRAIN_TIME_S / 60:.1f} นาที")
print(f"VRAM peak : {VRAM_PEAK_GB:.2f} GB")

LOG = [dict(h) for h in trainer.state.log_history]
reward_logs = [h for h in LOG if "reward" in h]
if reward_logs:
    first, last = reward_logs[0], reward_logs[-1]
    print("\nจาก log ของ TRL (step แรก -> step สุดท้าย):")
    for key in ("reward", "reward_std", "kl", "completion_length"):
        if key in first and key in last:
            print(f"  {key:18s} {first[key]:+8.4f} -> {last[key]:+8.4f}")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.2 -- วิเคราะห์จาก tracker: mean reward, สัดส่วนกลุ่มที่ std > 0, ความยาว
# -----------------------------------------------------------------------------
n_rounds = min(len(TRACK["correct"]), len(TRACK["format"]), len(TRACK["thai"]),
               len(TRACK["lens"]))
assert n_rounds > 0, "tracker ว่างเปล่า -- trainer ไม่ได้เรียก reward function เลยหรือ?"

ROUND_STATS = []
for i in range(n_rounds):
    totals = [a + b + c for a, b, c in zip(TRACK["correct"][i],
                                           TRACK["format"][i],
                                           TRACK["thai"][i])]
    groups = [totals[j:j + G] for j in range(0, len(totals), G)]
    frac_nonzero = sum(1 for g in groups if len(set(g)) > 1) / len(groups)
    ROUND_STATS.append({
        "round": i + 1,
        "mean_reward": sum(totals) / len(totals),
        "frac_groups_std_gt0": frac_nonzero,
        "mean_completion_tokens": sum(TRACK["lens"][i]) / len(TRACK["lens"][i]),
        "mean_correct": sum(TRACK["correct"][i]) / len(TRACK["correct"][i]),
    })

xs = [s["round"] for s in ROUND_STATS]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
axes[0].plot(xs, [s["mean_reward"] for s in ROUND_STATS], "-o", ms=3, color="#2563eb")
axes[0].set_title("mean reward ต่อรอบ generate", fontsize=10)
axes[1].plot(xs, [s["frac_groups_std_gt0"] for s in ROUND_STATS], "-o", ms=3, color="#16a34a")
axes[1].set_ylim(-0.05, 1.05)
axes[1].axhline(0, color="#dc2626", lw=1, ls="--")
axes[1].set_title("สัดส่วนกลุ่มที่ std > 0 (สัญญาณชีพ -- แตะศูนย์ = หยุดเรียน)", fontsize=10)
axes[2].plot(xs, [s["mean_completion_tokens"] for s in ROUND_STATS], "-o", ms=3, color="#f59e0b")
axes[2].set_title("ความยาว completion เฉลี่ย (token)", fontsize=10)
for ax in axes:
    ax.set_xlabel("รอบ generate")
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)
fig.suptitle("สามเส้นที่ต้องดูคู่กันเสมอใน GRPO", fontsize=11)
fig.tight_layout()
fig.savefig("fig_grpo_training.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"รอบแรก : reward={ROUND_STATS[0]['mean_reward']:.3f}  "
      f"std>0 {100 * ROUND_STATS[0]['frac_groups_std_gt0']:.0f}% ของกลุ่ม  "
      f"len={ROUND_STATS[0]['mean_completion_tokens']:.0f}")
print(f"รอบสุดท้าย: reward={ROUND_STATS[-1]['mean_reward']:.3f}  "
      f"std>0 {100 * ROUND_STATS[-1]['frac_groups_std_gt0']:.0f}% ของกลุ่ม  "
      f"len={ROUND_STATS[-1]['mean_completion_tokens']:.0f}")

print()
if HACK_ATTEMPTS:
    print(f"ดักได้! ความพยายามฟาร์ม format ด้วย <think></think> ว่างเปล่า: "
          f"{len(HACK_ATTEMPTS)} ครั้ง (ตัวอย่าง)")
    print("  " + HACK_ATTEMPTS[0][:160].replace(chr(10), " "))
    print("โชคดีที่ regex ของเราเป็น .+? มันจึงไม่ได้ +0.3 ไปฟรี ๆ -- แต่มันพยายามจริง")
else:
    print("รันนี้ไม่พบ <think></think> ว่างเปล่าเลย -- regex .+? ปิดช่องฟาร์มตั้งแต่ต้น")
    print("(export เป็น 0 ครั้งใน results.json ตามจริง -- การไม่เจอก็เป็นข้อมูล)")


---

## 9. เปรียบเทียบ (Comparison)

คำถามเดียวที่หัวข้อนี้ต้องตอบ: **GRPO "สร้าง" ความสามารถใหม่ หรือแค่ "เหลา" ของเดิม**

ตรรกะเบื้องหลังแข็งแรงกว่าที่เห็น: GRPO เรียนจาก advantage ซึ่งไม่เป็นศูนย์ได้
ก็ต่อเมื่อ**มีอย่างน้อยหนึ่งคำตอบในกลุ่มที่ทำได้ดีกว่าเพื่อน** — แปลว่าโจทย์ที่โมเดล
สุ่มยังไงก็ไม่เคยถูกเลย ($p = 0$) จะไม่มีวันส่งสัญญาณเรียนรู้เข้ามาในระบบ
สิ่งที่ RLVR ทำได้ดีคือ**ย้ายความสามารถที่กระจัดกระจายอยู่ใน pass@8 มากระจุกที่ pass@1**
(งานปี 2025 ของ Yue และคณะ วัดพบด้วยซ้ำว่าที่ $k$ ใหญ่มาก ๆ โมเดลฐานอาจ*ชนะ*
โมเดลหลัง RL) — นี่ไม่ได้แปลว่า GRPO ไร้ค่า: ผู้ใช้จริงได้คำตอบเดียว **pass@1 คือของจริง**

วิธีอ่านผลด้านล่าง:

- ถ้า pass@1 ขยับขึ้นมากกว่า pass@8 → พฤติกรรม "เหลา" ตามตำรา (คาดว่าจะเห็นแบบนี้)
- ถ้า pass@8 ขยับขึ้นด้วยพอ ๆ กัน → มีการสร้างเส้นทางใหม่จริง (เกิดได้ แต่อย่าคาดหวังที่สเกลนี้)
- ช่วง CI ที่ซ้อนกัน = อย่าเพิ่งสรุปอะไรทั้งนั้น (12 ข้อ × 8 คำตอบ คือ n เล็กมาก)

จากนั้นวัด KobEval-TH ซ้ำด้วยสัญญาเดิม แล้วปิดด้วยตารางมาตรฐานของซีรีส์ + McNemar


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.1 -- pass@k หลังเทรน บน held-out เดิม เงื่อนไขเดิมทุกประการ
# -----------------------------------------------------------------------------
sampled_after = sample_solutions(heldout_items, tag="pass@k หลังเทรน")
PASSK_AFTER = passk_report(heldout_items, sampled_after)

b1, a1 = PASSK_BEFORE["pass1"], PASSK_AFTER["pass1"]
bn, an = PASSK_BEFORE["passn"], PASSK_AFTER["passn"]

print()
print(f"{'':12s} {'pass@1':>10s} {'pass@' + str(PASSK_N):>10s} {'gap':>8s}")
print("-" * 44)
print(f"{'ก่อนเทรน':12s} {b1:10.3f} {bn:10.3f} {bn - b1:8.3f}")
print(f"{'หลังเทรน':12s} {a1:10.3f} {an:10.3f} {an - a1:8.3f}")
print(f"{'ส่วนต่าง':12s} {a1 - b1:+10.3f} {an - bn:+10.3f}")
print()
if (a1 - b1) > (an - bn):
    print("pass@1 ขยับมากกว่า pass@8 -> พฤติกรรม 'เหลา' (sharpening) ตามที่ทฤษฎีคาด")
else:
    print("pass@8 ขยับไม่น้อยกว่า pass@1 -> อ่านผลอย่างระวัง n เล็กมาก อย่าเพิ่งประกาศการค้นพบ")

fig, ax = plt.subplots(figsize=(7.5, 4.3))
width = 0.35
ks = [f"pass@1", f"pass@{PASSK_N}"]
before_vals = [b1, bn]
after_vals = [a1, an]
x = np.arange(len(ks))
err_b = [[b1 - PASSK_BEFORE["wilson_pass1"][0]], [PASSK_BEFORE["wilson_pass1"][1] - b1]]
err_a = [[a1 - PASSK_AFTER["wilson_pass1"][0]], [PASSK_AFTER["wilson_pass1"][1] - a1]]
ax.bar(x - width / 2, before_vals, width, color="#94a3b8", label="ก่อน GRPO")
ax.bar(x + width / 2, after_vals, width, color="#2563eb", label="หลัง GRPO")
ax.errorbar([x[0] - width / 2], [b1], yerr=err_b, fmt="none", ecolor="#334155", capsize=4)
ax.errorbar([x[0] + width / 2], [a1], yerr=err_a, fmt="none", ecolor="#334155", capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(ks)
ax.set_ylim(0, 1.0)
ax.set_ylabel("อัตราแก้โจทย์สำเร็จ")
ax.set_title(f"pass@1 vs pass@{PASSK_N} บน held-out {len(heldout_items)} ข้อ\n"
             "(error bar = Wilson 95% CI ของ pass@1 -- ถ้าซ้อนกันอย่าเพิ่งสรุป)", fontsize=10)
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_passk.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.2 -- วัด KobEval-TH ซ้ำด้วยสัญญาเดิม แล้วเทียบกับ baseline
# -----------------------------------------------------------------------------
policy.eval()
t0 = time.time()
after = evaluate(
    policy,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name=f"Qwen3-0.6B + GRPO ({'FAST' if FAST_MODE else 'full'})",
    out_path="results_after.json",
    extra={"train_time_s": TRAIN_TIME_S, "vram_peak_gb": VRAM_PEAK_GB},
)
print(f"ใช้เวลาวัดผลหลัง GRPO: {time.time() - t0:.0f} วินาที\n")

table = compare(baseline, after, out_path="results_table.json")

plot_before_after(
    baseline,
    after,
    slices=KOBEVAL_SLICES,
    title="ตอนที่ 5: ก่อน vs หลัง GRPO (TH-MATH)",
    out_path="fig_before_after.png",
    results_json="results_before_after.json",
)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.3 -- export: samples.json (เทียบก่อน-หลัง) + results.json (ตัวเลขทั้งหมด)
# -----------------------------------------------------------------------------
from kobeval import write_results

# --- samples.json: คำตอบ greedy ก่อนเทรน (เก็บไว้ในหัวข้อ 7.3) vs หลังเทรน --------
policy.eval()
sample_after_texts = []
with torch.no_grad():
    for it in SAMPLE_PROBLEMS:
        enc = tokenizer(it["prompt"], return_tensors="pt").to(DEV)
        torch.manual_seed(SEED)
        out = policy.generate(**enc, do_sample=False, max_new_tokens=MAX_COMPLETION,
                              pad_token_id=tokenizer.pad_token_id)
        sample_after_texts.append(
            tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

samples = []
for it, b_text, a_text in zip(SAMPLE_PROBLEMS, sample_before_texts, sample_after_texts):
    samples.append({
        "prompt": it["question"],
        "outputs": {"before": b_text, "after": a_text},
        "metrics": {
            "before": {
                "tokens": len(tokenizer(b_text, add_special_tokens=False).input_ids),
                "thai_ratio": round(th_ratio(b_text), 3),
                "correct": int(grpo_extract_final_int(b_text) == it["answer"]),
            },
            "after": {
                "tokens": len(tokenizer(a_text, add_special_tokens=False).input_ids),
                "thai_ratio": round(th_ratio(a_text), 3),
                "correct": int(grpo_extract_final_int(a_text) == it["answer"]),
            },
        },
    })

with open("samples.json", "w", encoding="utf-8") as f:
    json.dump({"items": samples}, f, ensure_ascii=False, indent=2)
print(f"เขียน samples.json แล้ว ({len(samples)} ข้อ)")

# --- results.json: ทุกตัวเลขของตอนนี้ในไฟล์เดียว ให้วิดเจ็ตบนบล็อกอ่านค่าจริง -----
payload = {
    "post": 5,
    "slug": "llm-05-grpo",
    "notebook": "05_grpo.ipynb",
    "model": MODEL_ID,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "datasets": ["VISAI-AI/gsm8k-thai"],
    "adapter_source": ADAPTER_SOURCE,
    "fast_mode": FAST_MODE,
    "hyperparameters": {
        "num_generations": G,
        "max_completion_length": MAX_COMPLETION,
        "max_prompt_length": 256,
        "temperature": 1.0,
        "beta_kl": BETA_KL,
        "epsilon": EPSILON,
        "learning_rate": LR,
        "per_device_train_batch_size": BATCH_COMPLETIONS,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "num_train_epochs": 1,
        "n_train_prompts": N_TRAIN,
        "rewards": {"correct": 1.0, "format": 0.3, "thai": 0.2},
        "fp16": USE_FP16,
        "lora_r": 16,
        "lora_alpha": 32,
    },
    "budget": {
        "total_completions": N_TRAIN * G,
        "completions_per_step": BATCH_COMPLETIONS * GRAD_ACCUM,
        "optimizer_steps": (N_TRAIN * G) // (BATCH_COMPLETIONS * GRAD_ACCUM),
        "max_generated_tokens": N_TRAIN * G * MAX_COMPLETION,
    },
    "training": {
        "train_time_s": TRAIN_TIME_S,
        "vram_peak_gb": VRAM_PEAK_GB,
        "round_stats": ROUND_STATS,
        "log_history": LOG,
    },
    "format_hack": {
        "n_attempts": len(HACK_ATTEMPTS),
        "examples": [h[:300] for h in HACK_ATTEMPTS],
        "note": "ความพยายามใช้ <think></think> ว่างเปล่า -- regex .+? ทำให้ไม่ได้คะแนน",
    },
    "passk": {
        "n_heldout": len(heldout_items),
        "n_samples_per_problem": PASSK_N,
        "before": PASSK_BEFORE,
        "after": PASSK_AFTER,
    },
    "kobeval": {
        "slices_run": KOBEVAL_SLICES,
        "before": {
            "model": baseline["model"],
            "accuracy": {k: v["accuracy"] for k, v in baseline["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in baseline["slices"].items()},
            "th_ratio": baseline["overall"]["th_ratio"],
        },
        "after": {
            "model": after["model"],
            "accuracy": {k: v["accuracy"] for k, v in after["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in after["slices"].items()},
            "th_ratio": after["overall"]["th_ratio"],
        },
        "mcnemar": table.get("mcnemar"),
        "markdown_table": table["markdown"],
    },
    "figures": [
        "fig_group_advantage.png",
        "fig_k3_estimator.png",
        "fig_grpo_training.png",
        "fig_passk.png",
        "fig_before_after.png",
    ],
}

path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}")
print(json.dumps({"passk": {"before": {"pass1": PASSK_BEFORE["pass1"],
                                        "passn": PASSK_BEFORE["passn"]},
                             "after": {"pass1": PASSK_AFTER["pass1"],
                                       "passn": PASSK_AFTER["passn"]}},
                  "format_hack_attempts": len(HACK_ATTEMPTS)},
                 ensure_ascii=False, indent=2))


---

## 10. สรุป (Summary)

- **ค่าเฉลี่ยของกลุ่มคือ baseline ที่ไม่ต้องเทรน** — สุ่ม $G$ คำตอบจาก prompt เดียวกัน
  แล้ว value network ของ PPO ทั้งตัวก็ไม่จำเป็นอีกต่อไป
- **Reward ที่ตรวจได้ด้วยโค้ด = ศูนย์ label มนุษย์** — โจทย์เลขตรวจด้วย `==` บรรทัดเดียว
- **Advantage เป็นเรื่องสัมพัทธ์ภายในกลุ่ม** — คำตอบ reward บวกยังโดนผลักลงได้
  ถ้าเพื่อนร่วมกลุ่มทำได้ดีกว่า (เราเห็นกับตาในหัวข้อ 7.2)
- **กลุ่มที่ reward เท่ากันหมดสอนอะไรไม่ได้เลย** — `advantage = 0 → batch นี้ไม่สอนอะไรเลย`
  plot สัดส่วนกลุ่มที่ std > 0 เสมอ มันคือเส้นแบ่งระหว่าง "อิ่มตัว" กับ
  "หยุดเรียนไปนานแล้วโดยไม่มีใครรู้"
- **k3 ทำให้ KL penalty ใช้งานได้จริงที่ batch เล็ก** — unbiased เท่า k1
  แต่ไม่ติดลบและ variance ต่ำกว่ากันหลายเท่า (พิสูจน์ด้วยการจำลองในหัวข้อ 4)
- **หารด้วย std มี bias ซ่อนอยู่** — Dr.GRPO ตัดทิ้งเหลือแค่ลบ mean
  (ใน TRL คือ `scale_rewards=False`)
- **Reward shaping จำเป็นแต่อันตราย** — reward ย่อยกันกลุ่ม all-zero ช่วงแรก
  แต่เปิดช่องให้ฟาร์มคะแนน โมเดลไม่ได้ optimize สิ่งที่คุณตั้งใจ
  มัน optimize สิ่งที่คุณเขียน
- **RLVR ส่วนใหญ่ "เหลา" ไม่ใช่ "สร้าง"** — pass@1 ไต่เข้าหาเพดาน pass@8 เดิม
  วัดทั้งคู่เสมอ

**ตอนต่อไป:** Context Distillation — system prompt ยาวเหยียดที่ต้องจ่ายทุกครั้ง
ที่เรียกโมเดล เอามา**กลั่นใส่น้ำหนัก**ให้โมเดลประพฤติตัวตามนั้น
โดยไม่ต้องเห็น prompt อีกเลยได้อย่างไร


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **GRPO ต้องการ reward ที่ตรวจได้ด้วยโค้ด** — โจทย์เลขตรวจได้ โค้ดตรวจได้ (รัน test)
   แต่ "เขียนอีเมลภาษาไทยให้สุภาพเป็นธรรมชาติ" ไม่มีฟังก์ชันตรวจ — งานปลายเปิดแบบนั้น
   คือดินแดนของ preference data และ DPO จากตอนที่ 4 สองตอนนี้**เสริมกัน ไม่ได้แทนกัน**
   เลือกเครื่องมือจากรูปร่างของ reward ไม่ใช่จากความใหม่ของอัลกอริทึม

2. **อย่าตีความผลว่าโมเดล "ฉลาดขึ้น"** — หลักฐานทั้งของเราและของงานวิจัยที่สเกลจริง
   ชี้ว่า RLVR ส่วนใหญ่จัดระเบียบความน่าจะเป็นของความสามารถที่มีอยู่แล้วที่ pass@8
   ให้มาโผล่ที่ pass@1 อย่างเสถียร ถ้าอยากได้ความรู้ใหม่จริง ๆ ต้องกลับไปที่ CPT (ตอนที่ 1)

3. **FAST_MODE คือค่าเริ่มต้น และมันเล็กกว่า config ที่รายงานในบทความ** —
   64 โจทย์ × 4 คำตอบ × 8 step คือการสาธิตกลไก ไม่ใช่การเทรนจริง
   config เต็ม (128 × 8, 16 step) เปิดได้ด้วย `FAST_MODE = False`
   ส่วน DeepSeek-R1 ใช้โจทย์ระดับแสนข้อ — ต่างกันหลาย order of magnitude

4. **held-out เล็กมาก (12 ข้อ × 8 คำตอบ)** — Wilson CI จะกว้าง ความต่างเล็ก ๆ
   ของ pass@1 ก่อน-หลัง **ไม่ใช่** หลักฐานที่มีนัยสำคัญ และ KobEval-TH slice ละ 30 ข้อ
   ให้ดูค่า p จาก McNemar ที่ `compare()` พิมพ์เสมอ

5. **รันครั้งเดียว seed เดียว** — RL online มีความแปรปรวนสูงกว่าการเทรนแบบ supervised
   มาก ผลของ seed เดียวบอกทิศทางได้ แต่บอกขนาดของผลไม่ได้

6. **reward ตรวจเฉพาะคำตอบสุดท้าย ไม่ตรวจวิธีทำ** — โมเดลที่เดาถูกกับโมเดลที่คิดถูก
   ได้ reward เท่ากัน ที่สเกลนี้เราแยกสองแบบนั้นไม่ออก

7. **ฮาร์ดแวร์จำกัด** — ทุกอย่างถูกบีบให้รันได้บน T4 ฟรี: fp16 แทน bf16,
   sdpa แทน FlashAttention-2, ไม่มี vLLM ช่วย generate (ซึ่งระบบ production ใช้กัน)
   ผลบนฮาร์ดแวร์ที่ใหญ่กว่าอาจต่างออกไป

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต Apache-2.0 — [kobkrit.com](https://kobkrit.com)*
